<a href="https://colab.research.google.com/github/alperarslan19/asthma-voice-dl/blob/main/notebooks/27_background_step0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 27 — Arka plan (ortam sesi) Adım 0: çıkarma, kalite kapısı, bilgi testi (EXP-032; CPU)

**Bilimsel amaç:** Kayıt bağlamı etiketi neredeyse belirliyor; açık kalan yollardan biri **ortam** (G2). Burada, ileride yapılacak arka plan değiştirme testinin ön koşulu olarak üç soruyu sırayla soruyoruz (`docs/BACKGROUND_DESIGN.md`, D-046):
- **0a** Kayıtlardan arka plan çıkarılabiliyor mu? Kişi başına kaç saniye, gruplara göre dengeli mi?
- **0b** Çıkan şey gerçekten arka plan mı? Konuşma sızıntısı testi (cinsiyet), A1–B spektrum tutarlılığı, **kör dinleme** (Alper). Bu adım astım etiketiyle başarım ÖLÇMEZ; parametreler burada dondurulur.
- **0c** Arka plan tek başına etiketi / dönemi ayırıyor mu? Yalnız **D-046 KABUL**'den sonra (script kendisi kontrol eder).

Üç arka plan türü: **A1** oda tonu (konuşmadan ≥ 150 ms uzak, tabana ≤ 3 dB yakın), **A2** tüm sessizlik (sesli konuşmadan uzak; nefes dahil), **B** en sessiz karelerden durağan spektrum + sentez. Hepsi modelin gördüğü biçimde (önbellekle aynı zincir).

GPU gerekmez: *Runtime → CPU*. Süre: 0a ~15–30 dk, 0b ~5 dk [tahmin]. Raporlar yalnız agrega; katılımcı düzeyi dosyalar Drive `experiments/background_v1/`.

**Bu notebook iki seferde çalışır:** önce Bölüm 0–4 (0a + 0b) → dinleme formunu doldur → PR birleşsin ve D-046 KABUL olsun → Bölüm 5 (0c).

## 0. Oturum kurulumu

In [ ]:
import shutil, subprocess, platform, sys
print('Python:', sys.version.split()[0], '|', platform.platform())
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                         capture_output=True, text=True).stdout)
else:
    print('UYARI: GPU yok (CPU oturumu). Bu notebook CPU üzerinde çalışır; sorun değil.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess, sys, os, json, shutil, hashlib, codecs, platform
DRIVE_ROOT  = Path('/content/drive/MyDrive/asthma-voice')
RAW, DERIVED = DRIVE_ROOT / 'data_raw', DRIVE_ROOT / 'data_derived'
EXPERIMENTS = DRIVE_ROOT / 'experiments'
AUDIO_ZIP   = RAW / 'soundData.zip'
AUDIO_DIR   = Path('/content/audio')                       # zip YEREL diske açılır
DRIVE_CACHE = DERIVED / 'audio_cache_v1'
SPLITS_DAY  = DERIVED / 'splits_day'
SPEECH_MFCC = DERIVED / 'mfcc' / 'features_cache.csv'     # cinsiyet pozitif kontrolü (EXP-011 özellikleri)
BG_LOCAL    = Path('/content/background_v1')              # önce yerelde üretilir (Drive'a çok küçük yazma yavaş)
BG_DRIVE    = EXPERIMENTS / 'background_v1'               # sonra Drive'a kopyalanır (katılımcı düzeyi → git'e GİRMEZ)
GITHUB_USER, REPO_NAME = 'alperarslan19', 'asthma-voice-dl'
REPO_DIR    = Path('/content') / REPO_NAME
EXPECTED_N_RECORDINGS, EXPECTED_N_PARTICIPANTS = 2393, 342   # Faz 1 önbellek kohortu (reports/audio_cache)
for p in [AUDIO_ZIP, DERIVED / 'recording_map.csv', DERIVED / 'participant_context.csv', DRIVE_CACHE / 'index.csv',
          SPLITS_DAY / 'outer_r0.csv']:
    assert p.exists(), f'Bulunamadı: {p}'
if not SPEECH_MFCC.exists():
    print('UYARI: features_cache.csv yok → cinsiyet pozitif kontrolü atlanır (kapı yine hesaplanır).')


def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def sh(cmd, check=True):
    """Kabuk komutunu çalıştırır, çıktıyı canlı yazar. check=True iken komut başarısızsa hücreyi DURDURUR.
    (`!komut` başarısız olsa da hücre devam eder; 'Run all' hatayı fark etmeden sonraki adıma geçerdi.)"""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    dec = codecs.getincrementaldecoder('utf-8')('replace')
    while True:
        b = p.stdout.read1(1 << 14)
        if not b:
            break
        sys.stdout.write(dec.decode(b))
        sys.stdout.flush()
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f'Komut başarısız (çıkış kodu {rc}): {cmd[:300]}')
    return None if check else rc


print('Girdiler mevcut.')

In [ ]:
# Repo: yoksa klonla; varsa GitHub'daki main'e güvenli biçimde geç (önceki çalıştırmanın rapor dalında kalmış olabilir)
from google.colab import userdata
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception as e:
    raise RuntimeError("GITHUB_TOKEN okunamadı: sol paneldeki anahtar (Secrets) simgesinden GITHUB_TOKEN ekli mi ve "
                       "'Notebook access' açık mı?") from e
assert token, 'GITHUB_TOKEN boş'
REMOTE = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'


def git(*args, check=True, show=False):
    """git komutu; çıktıda token gizlenir. check=True iken hata durumunda hücreyi durdurur."""
    r = subprocess.run(['git', '-C', str(REPO_DIR), *args], capture_output=True, text=True)
    r.stdout, r.stderr = r.stdout.replace(token, '***'), r.stderr.replace(token, '***')
    if show and (r.stdout + r.stderr).strip():
        print((r.stdout + r.stderr).strip())
    if check and r.returncode != 0:
        cmd = ' '.join(args[:3]).replace(token, '***')
        raise RuntimeError(f'git {cmd} başarısız (çıkış {r.returncode}):\n{(r.stdout + r.stderr).strip()}')
    return r


if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    raise RuntimeError(f'{REPO_DIR} var ama git deposu değil (yarım kalmış klon?). Şunu çalıştırıp bu hücreyi yeniden '
                       f'çalıştır:  !rm -rf {REPO_DIR}')
if not REPO_DIR.exists():
    r = subprocess.run(['git', 'clone', REMOTE, str(REPO_DIR)], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone başarısız:\n' + (r.stdout + r.stderr).replace(token, '***'))
    print('Klonlandı.')
else:
    git('remote', 'set-url', 'origin', REMOTE)            # token yenilendiyse
    git('fetch', '--prune', 'origin')
    branch = git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip()
    unpushed = git('log', 'HEAD', '--not', '--remotes', '--oneline').stdout.strip()
    dirty = git('status', '--porcelain', '--untracked-files=all').stdout.strip()
    if unpushed:
        raise RuntimeError(
            f"'{branch}' dalında GitHub'a gönderilmemiş commit var:\n{unpushed}\n"
            "Bu, yarım kalmış bir rapor gönderimidir; raporu kaybetmemek için önce 'Raporu GitHub'a gönder' hücresini "
            "çalıştır (aynı commit'i yeniden gönderir), sonra bu hücreyi yeniden çalıştır.")
    if dirty:
        at_main = branch == 'main' and git('rev-parse', 'HEAD').stdout == git('rev-parse', 'origin/main').stdout
        if not at_main:
            raise RuntimeError(
                f"Repo klasöründe commit edilmemiş dosyalar var (dal: {branch}):\n{dirty}\n"
                "Bunlar önceki bir çalıştırmanın raporlarıysa önce gönder hücresini çalıştır. Gerekmiyorsa klasörü silip "
                f"bu hücreyi yeniden çalıştır:  !rm -rf {REPO_DIR}")
        print('UYARI: main zaten GitHub ile aynı; önceki çalıştırmadan kalan commit edilmemiş dosyalara dokunulmadı:\n' + dirty)
    else:
        git('checkout', '-B', 'main', 'origin/main')       # temiz klasör: main = GitHub'daki main (yerel diğer dallar kalır)
        if branch != 'main':
            print(f"'{branch}' dalından main'e geçildi (o dal GitHub'da duruyor).")
assert git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip() == 'main'
COMMIT = git('rev-parse', 'HEAD').stdout.strip()
assert COMMIT == git('rev-parse', 'origin/main').stdout.strip(), 'yerel main GitHub main ile aynı değil'
print('Dal: main | commit:', COMMIT[:12], '(= origin/main)')

In [ ]:
sh(f'pip install -q -r {REPO_DIR}/requirements-colab.txt')
import numpy as np, pandas as pd, sklearn, statsmodels
print('numpy', np.__version__, '| pandas', pd.__version__, '| sklearn', sklearn.__version__, '| statsmodels', statsmodels.__version__)

In [ ]:
# Kapı: kod ve tasarım main'de
for p in ['docs/BACKGROUND_DESIGN.md', 'scripts/background_extract.py', 'tests/test_background.py']:
    if git('ls-files', '--error-unmatch', p, check=False).returncode != 0:
        raise RuntimeError(f'{p} main dalında yok: PR birleştirildi mi? Sonra 0. bölümü yeniden çalıştır.')
assert '## D-046' in (REPO_DIR / 'DECISIONS.md').read_text(), 'D-046 main dalında yok'
sh(f'cd {REPO_DIR} && python tests/test_background.py')     # birim + uçtan uca test (sentetik; ~3–5 dk)
print('Kapı geçti.')

## 1. Ses dosyalarını aç (yerel disk)

In [ ]:
import zipfile, time
t0 = time.time()
if not AUDIO_DIR.exists():
    shutil.copy(AUDIO_ZIP, '/content/soundData.zip')
    with zipfile.ZipFile('/content/soundData.zip') as z:
        z.extractall(AUDIO_DIR)
n = sum(1 for _ in AUDIO_DIR.rglob('*.m4a'))
print(f'.m4a: {n} ({time.time() - t0:.0f} s)')
assert n >= EXPECTED_N_RECORDINGS, f'beklenen ≥ {EXPECTED_N_RECORDINGS}, bulunan {n}'
import pandas as pd
idx = pd.read_csv(DRIVE_CACHE / 'index.csv')
assert len(idx) == EXPECTED_N_RECORDINGS and idx.participant_id.nunique() == EXPECTED_N_PARTICIPANTS, 'önbellek dizini beklenen kohort değil'
print('Önbellek dizini:', len(idx), 'kayıt,', idx.participant_id.nunique(), 'katılımcı')

## 2. Adım 0a — Arka plan çıkarma ve envanter
Önce 5 katılımcıyla **smoke test** (şekil, kazanç uyumu, süre), sonra tam çalıştırma. Çıktılar önce yerel diske, sonra Drive'a.

In [ ]:
SMOKE = Path('/content/bg_smoke')
sh(f'cd {REPO_DIR} && python scripts/background_extract.py extract --audio-root {AUDIO_DIR} '
   f'--recording-map {DERIVED}/recording_map.csv --cache-index {DRIVE_CACHE}/index.csv '
   f'--context {DERIVED}/participant_context.csv --out-dir {SMOKE} --report-dir {SMOKE}/rep --n-listen 2 --limit 5')
ft = pd.read_csv(SMOKE / 'features.csv'); st = pd.read_csv(SMOKE / 'rec_stats.csv')
assert ft.groupby('participant_id').size().eq(3).all() and len(st) >= 30
assert st.gain_diff_db.abs().max() < 0.1, 'model görünümü önbellekle aynı kazancı vermiyor'
print(st[['snr_db', 'A1_s', 'A2_s']].describe().round(2))
print('Smoke test geçti.')

In [ ]:
sh(f'cd {REPO_DIR} && python scripts/background_extract.py extract --audio-root {AUDIO_DIR} '
   f'--recording-map {DERIVED}/recording_map.csv --cache-index {DRIVE_CACHE}/index.csv '
   f'--context {DERIVED}/participant_context.csv --out-dir {BG_LOCAL} --report-dir {REPO_DIR}/reports/background --n-listen 30')
shutil.copytree(BG_LOCAL, BG_DRIVE, dirs_exist_ok=True)
print('Drive\'a kopyalandı:', BG_DRIVE)
print((REPO_DIR / 'reports/background/BG-0a_inventory.md').read_text())

## 3. Adım 0b — Konuşma sızıntısı kapısı (cinsiyet)

In [ ]:
pos = f'--speech-features {SPEECH_MFCC}' if SPEECH_MFCC.exists() else ''
sh(f'cd {REPO_DIR} && python scripts/background_extract.py probe --features {BG_LOCAL}/features.csv '
   f'--context {DERIVED}/participant_context.csv --splits-dir {SPLITS_DAY} --tasks sex {pos} '
   f'--out-dir {BG_LOCAL} --report-dir {REPO_DIR}/reports/background')
shutil.copytree(BG_LOCAL, BG_DRIVE, dirs_exist_ok=True)
print((REPO_DIR / 'reports/background/BG-0b_qc.md').read_text())

### Kör dinleme (Alper)
1. Drive'da `experiments/background_v1/listening/` klasörünü indir: `L01a.wav` … ve `listening_sheet.csv`.
2. **`listening_key.csv` dosyasını AÇMA** (hangi harfin hangi tür olduğunu gösterir; körlük bozulur).
3. Her dosyayı dinle; formda 0 / 1 / 2 puan ver: konuşma kalıntısı, nefes, tık / bozulma, döngü hissi (0 = yok, 1 = hafif, 2 = belirgin). Seviyeler dinleme için eşitlendi; gürültünün yüksekliğini puanlama.
4. Doldurulmuş formu aynı klasöre **`listening_sheet_filled.csv`** adıyla yükle.

## 4. Raporları GitHub'a gönder (0a + 0b)

In [ ]:
# Rapor commit'i her seferinde YENİ, zaman damgalı bir dala gider (aynı adlı dala ikinci gönderim reddedilirdi).
# Gönderim başarısız olursa commit yerel dalda kalır; bu hücreyi yeniden çalıştırmak aynı commit'i yeniden gönderir.
import re, time
import pandas as pd
BASE_BRANCH, PATHS, MSG = 'background-0ab', ['reports/background/BG-0a_inventory.md', 'reports/background/BG-0a_inventory.json', 'reports/background/BG-0b_qc.md', 'reports/background/BG-0b_qc.json'], 'EXP-032 Adım 0a–0b: arka plan envanteri ve sızıntı kapısı (Colab)'
git('config', 'user.name', GITHUB_USER)
git('config', 'user.email', f'{GITHUB_USER}@users.noreply.github.com')
cur = git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip()
if cur.startswith(BASE_BRANCH + '-') and git('log', 'HEAD', '--not', '--remotes', '--oneline').stdout.strip():
    BRANCH = cur
    print('Önceki denemenin commit\'i yeniden gönderiliyor:', BRANCH)
else:
    assert cur == 'main', f"Beklenmeyen dal '{cur}': 0. bölümdeki repo hücresini yeniden çalıştır."
    git('add', '--', *PATHS)
    staged = git('diff', '--cached', '--name-only').stdout.split()
    assert staged, 'Gönderilecek değişiklik yok (rapor üretilmedi mi? yukarıdaki hücrenin çıktısına bak).'
    bad = [s for s in staged if not any(s == p or s.startswith(p.rstrip('/') + '/') for p in PATHS)]
    assert not bad, f'Beklenmeyen dosya: {bad}'
    ids = set(pd.read_csv(DERIVED / 'participant_context.csv').participant_id.astype(int))   # D-014 son kontrol
    for s in staged:
        txt = (REPO_DIR / s).read_text(errors='ignore')
        found = {int(m) for m in re.findall(r'(?<![\d.])(\d{6})(?![\d.])', txt)} & ids
        assert token not in txt and not found, f'{s}: token ya da katılımcı ID\'si içeriyor → commit edilmedi (D-014)'
    BRANCH = f"{BASE_BRANCH}-{time.strftime('%Y%m%d-%H%M%S', time.gmtime())}"
    git('checkout', '-b', BRANCH)
    git('commit', '-m', MSG)
    print('commit:', git('rev-parse', '--short', 'HEAD').stdout.strip(), '| dosyalar:', staged)
r = git('push', '-u', 'origin', BRANCH, check=False, show=True)
if r.returncode != 0:
    raise RuntimeError(f"Push başarısız. Commit yerel '{BRANCH}' dalında duruyor (kaybolmadı). Token/ağ sorununu giderip "
                       "BU hücreyi yeniden çalıştır; aynı commit yeniden gönderilir.")
git('checkout', 'main')
print(f'Gönderildi: {BRANCH}\nPR aç: https://github.com/{GITHUB_USER}/{REPO_NAME}/pull/new/{BRANCH}')

## 5. Adım 0c — Dinleme özeti + bilgi testi (ikinci oturum)
Ön koşullar: dinleme formu doldurulmuş; 0a / 0b sonuçları ve parametreler gözden geçirilip **D-046 KABUL** olarak main'e birleştirilmiş (script, KABUL değilse 0c'yi çalıştırmaz). Yeni oturumda önce Bölüm 0'ı çalıştır.

In [ ]:
import shutil
BG_LOCAL.mkdir(exist_ok=True)
for name in ['features.csv']:
    if not (BG_LOCAL / name).exists():
        shutil.copy(BG_DRIVE / name, BG_LOCAL / name)
filled = BG_DRIVE / 'listening' / 'listening_sheet_filled.csv'
assert filled.exists(), 'Dinleme formu yok: listening_sheet_filled.csv'
sh(f'cd {REPO_DIR} && python scripts/background_extract.py listening --sheet "{filled}" '
   f'--key "{BG_DRIVE}/listening/listening_key.csv" --report-dir {REPO_DIR}/reports/background')
print((REPO_DIR / 'reports/background/BG-0b_listening.md').read_text())

In [ ]:
sh(f'cd {REPO_DIR} && python scripts/background_extract.py probe --features {BG_LOCAL}/features.csv '
   f'--context {DERIVED}/participant_context.csv --splits-dir {SPLITS_DAY} --tasks label period '
   f'--out-dir {BG_LOCAL} --report-dir {REPO_DIR}/reports/background')
shutil.copytree(BG_LOCAL, BG_DRIVE, dirs_exist_ok=True)
print((REPO_DIR / 'reports/background/BG-0c_info.md').read_text())

In [ ]:
# Rapor commit'i her seferinde YENİ, zaman damgalı bir dala gider (aynı adlı dala ikinci gönderim reddedilirdi).
# Gönderim başarısız olursa commit yerel dalda kalır; bu hücreyi yeniden çalıştırmak aynı commit'i yeniden gönderir.
import re, time
import pandas as pd
BASE_BRANCH, PATHS, MSG = 'background-0c', ['reports/background/BG-0b_listening.md', 'reports/background/BG-0b_listening.json', 'reports/background/BG-0c_info.md', 'reports/background/BG-0c_info.json'], 'EXP-032 Adım 0c: dinleme özeti ve bilgi testi (Colab)'
git('config', 'user.name', GITHUB_USER)
git('config', 'user.email', f'{GITHUB_USER}@users.noreply.github.com')
cur = git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip()
if cur.startswith(BASE_BRANCH + '-') and git('log', 'HEAD', '--not', '--remotes', '--oneline').stdout.strip():
    BRANCH = cur
    print('Önceki denemenin commit\'i yeniden gönderiliyor:', BRANCH)
else:
    assert cur == 'main', f"Beklenmeyen dal '{cur}': 0. bölümdeki repo hücresini yeniden çalıştır."
    git('add', '--', *PATHS)
    staged = git('diff', '--cached', '--name-only').stdout.split()
    assert staged, 'Gönderilecek değişiklik yok (rapor üretilmedi mi? yukarıdaki hücrenin çıktısına bak).'
    bad = [s for s in staged if not any(s == p or s.startswith(p.rstrip('/') + '/') for p in PATHS)]
    assert not bad, f'Beklenmeyen dosya: {bad}'
    ids = set(pd.read_csv(DERIVED / 'participant_context.csv').participant_id.astype(int))   # D-014 son kontrol
    for s in staged:
        txt = (REPO_DIR / s).read_text(errors='ignore')
        found = {int(m) for m in re.findall(r'(?<![\d.])(\d{6})(?![\d.])', txt)} & ids
        assert token not in txt and not found, f'{s}: token ya da katılımcı ID\'si içeriyor → commit edilmedi (D-014)'
    BRANCH = f"{BASE_BRANCH}-{time.strftime('%Y%m%d-%H%M%S', time.gmtime())}"
    git('checkout', '-b', BRANCH)
    git('commit', '-m', MSG)
    print('commit:', git('rev-parse', '--short', 'HEAD').stdout.strip(), '| dosyalar:', staged)
r = git('push', '-u', 'origin', BRANCH, check=False, show=True)
if r.returncode != 0:
    raise RuntimeError(f"Push başarısız. Commit yerel '{BRANCH}' dalında duruyor (kaybolmadı). Token/ağ sorununu giderip "
                       "BU hücreyi yeniden çalıştır; aynı commit yeniden gönderilir.")
git('checkout', 'main')
print(f'Gönderildi: {BRANCH}\nPR aç: https://github.com/{GITHUB_USER}/{REPO_NAME}/pull/new/{BRANCH}')